In [ ]:
import sys
sys.path.append("..")          
import pandas as pd
from src.data import load_pad_ufes_20
from src.train import get_device
from src.experiment import run_external

TAG = "medium_"                 
device = get_device()         

# Load PAD-UFES-20 with our four classes
df_ext = load_pad_ufes_20("../data/load_pad_ufes_20")
print(df_ext.shape)
print(df_ext["dx"].value_counts())              # images per diagnosis
print(df_ext["label"].value_counts())           # 0 = benign, 1 = malignant

(1376, 6)
dx
BCC    845
NEV    244
SEK    235
MEL     52
Name: count, dtype: int64
label
1    897
0    479
Name: count, dtype: int64


In [ ]:
# Read the internal results: they tell us which models exist and which cutoff each one used
internal = pd.read_csv(f"../results/{TAG}internal_results.csv")

rows = []
for _, r in internal.iterrows():
    print("External test:", r["config"], "seed", int(r["seed"]))
    # Test this saved model on PAD-UFES-20, with the cutoff chosen on HAM10000 validation
    rows.append(run_external(df_ext, r["config"], int(r["seed"]), r["threshold"], device, tag=TAG))

ext = pd.DataFrame(rows)
ext.to_csv(f"../results/{TAG}external_results.csv", index=False)

External test: A_baseline seed 0
External test: B_augment seed 0
External test: C_weights seed 0
External test: D_both seed 0
External test: A_baseline seed 1
External test: B_augment seed 1
External test: C_weights seed 1
External test: D_both seed 1
External test: A_baseline seed 2
External test: B_augment seed 2
External test: C_weights seed 2
External test: D_both seed 2


In [9]:
# Put internal and external scores side by side
merged = internal.merge(ext, on=["config", "seed"])

# Gap = internal score minus external score (a bigger number means a bigger drop)
metrics = ["auc", "balanced_acc", "sensitivity", "specificity"]
for m in metrics:
    merged[f"gap_{m}"] = merged[f"test_{m}"] - merged[f"ext_{m}"]

# Mean and standard deviation over the 3 seeds, per configuration
cols = [f"{p}_{m}" for m in metrics for p in ("test", "ext", "gap")]
stats = merged.groupby("config")[cols].agg(["mean", "std"])

# Compact table: each cell shows "mean ± std"
compact = pd.DataFrame(index=stats.index)
for c in cols:
    compact[c] = stats[(c, "mean")].round(3).astype(str) + " ± " + stats[(c, "std")].round(3).astype(str)

compact.T   

config,A_baseline,B_augment,C_weights,D_both
test_auc,0.849 ± 0.023,0.877 ± 0.028,0.844 ± 0.025,0.882 ± 0.019
ext_auc,0.703 ± 0.038,0.776 ± 0.025,0.746 ± 0.056,0.777 ± 0.026
gap_auc,0.146 ± 0.017,0.101 ± 0.037,0.097 ± 0.044,0.105 ± 0.032
test_balanced_acc,0.767 ± 0.021,0.785 ± 0.032,0.762 ± 0.01,0.79 ± 0.035
ext_balanced_acc,0.638 ± 0.035,0.697 ± 0.017,0.682 ± 0.046,0.695 ± 0.017
gap_balanced_acc,0.129 ± 0.034,0.088 ± 0.032,0.08 ± 0.036,0.095 ± 0.023
test_sensitivity,0.676 ± 0.058,0.813 ± 0.041,0.681 ± 0.038,0.841 ± 0.042
ext_sensitivity,0.56 ± 0.097,0.775 ± 0.038,0.597 ± 0.11,0.736 ± 0.121
gap_sensitivity,0.116 ± 0.091,0.038 ± 0.067,0.085 ± 0.082,0.105 ± 0.105
test_specificity,0.858 ± 0.033,0.757 ± 0.058,0.842 ± 0.022,0.738 ± 0.054
